In [36]:
# install python dependencies
!pip install mediapipe

# download tasks
!wget https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task


--2026-10-12 14:34:21--  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.181.207, 74.125.69.207, 173.194.193.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.181.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30664242 (29M) [application/octet-stream]
Saving to: ‘pose_landmarker_heavy.task.1’

pose_landmarker_hea 100%[===================>]  29.24M   142MB/s    in 0.2s    

2026-10-12 14:34:21 (142 MB/s) - ‘pose_landmarker_heavy.task.1’ saved [30664242/30664242]



In [27]:
import os
import time
import csv
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

from typing import Optional, Callable
from os import PathLike
from cv2 import VideoCapture
from pathlib import Path
from google.colab import drive
from mediapipe.tasks.python import BaseOptions
from mediapipe.tasks.python.vision import PoseLandmarkerOptions, PoseLandmarker, \
    RunningMode


In [2]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
dataset_dir = Path("/content/drive/MyDrive/datasets/workoutfitness-video")

print("----- Available exercises -----\n\n- ", end="")
print("\n- ".join(list(map(lambda path: Path(path).name, dataset_dir.iterdir()))))

----- Available exercises -----

- barbell biceps curl
- bench press
- chest fly machine
- deadlift
- decline bench press
- hammer curl
- hip thrust
- incline bench press
- lat pulldown
- lateral raise
- leg extension
- leg raises
- plank
- pull Up
- push-up
- romanian deadlift
- russian twist
- shoulder press
- squat
- t bar row
- tricep Pushdown
- tricep dips


In [4]:
# Set the exercise here from the above list

EXERCISE = "deadlift"

In [5]:
model_path = str(os.getcwd() / Path("pose_landmarker_heavy.task"))

options = PoseLandmarkerOptions(
  base_options=BaseOptions(model_asset_path=model_path),
  running_mode=RunningMode.VIDEO
)

landmarker = PoseLandmarker.create_from_options(options)

In [36]:
class VideoCaptureListeners:

    def __init__(
        self,
        on_start: Optional[Callable[[], None]] = None,
        on_frame: Optional[Callable[[np.ndarray], None]] = None,
        on_end: Optional[Callable[[], None]] = None,
    ):
        self.on_start = on_start
        self.on_frame = on_frame
        self.on_end = on_end

class VideoSource:

    def __init__(self, cap_source: str | PathLike[str] | int):
        self.buffer = []
        self.buffer_timestamps = []
        self.cap = VideoCapture(cap_source)
        if not self.cap.isOpened():
            raise IOError(f"Cannot open video source: {cap_source}")
        self._stop_requested = False
        self.replay_idx = 0

        fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.is_file_source = not isinstance(cap_source, int)
        self.source_fps = fps if (self.is_file_source and fps and fps > 0) else None

    @staticmethod
    def from_filepicker(filepath, *args, **kwargs):
        return VideoSource(filepath, *args, **kwargs)

    @staticmethod
    def from_camera(*args, **kwargs):
        return VideoSource(0, *args, **kwargs)

    def stop(self):
        self._stop_requested = True

    def next(self, reset=False):
        if self.replay_idx < 0:
            self.replay_idx = 0
        curr_frame = self.buffer[self.replay_idx]
        curr_timestamp = self.buffer_timestamps[self.replay_idx]

        self.replay_idx += 1
        if self.playback_is_empty() and reset:
            self.replay_idx = 0

        next_idx = self.replay_idx if self.replay_idx < len(self.buffer) else 0
        next_timestamp = self.buffer_timestamps[next_idx]

        return curr_frame, curr_timestamp, next_timestamp

    def reset_playback(self):
        self.replay_idx = 0

    def playback_is_empty(self):
        return self.replay_idx >= len(self.buffer)

    def capture_video(self, listeners: Optional[VideoCaptureListeners] = None):
        self._stop_requested = False
        if listeners and listeners.on_start: listeners.on_start()

        while not self._stop_requested:
            ret, frame = self.cap.read()
            if not ret:
                break

            self.buffer.append(frame)
            self.buffer_timestamps.append(time.monotonic_ns())
            if listeners and listeners.on_frame: listeners.on_frame(frame)

        self.cap.release()
        if listeners and listeners.on_end: listeners.on_end()
        return self.buffer


In [22]:
exercise_path = dataset_dir / EXERCISE

N_LANDMARKS = 33

def estimate(frame, timestamp):
  img = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  )
  return frame, landmarker.detect_for_video(img, timestamp)

def run_video_frames(videosource: VideoSource):
  video_estimations = []
  while not videosource.playback_is_empty():
    frame, timestamp, next_timestamp = videosource.next()
    frame, estimations = estimate(frame, timestamp // 1_000_000)
    video_estimations.append(estimations)
  return video_estimations



In [26]:
filename = EXERCISE + "-estimations.csv"
estimation_output_file = open(filename, "w", newline="")
writer = csv.writer(estimation_output_file)

for path in exercise_path.iterdir():
  videosource = VideoSource.from_filepicker(path)
  videosource.capture_video()
  estimations = run_video_frames(videosource)

  world_estimations = [
    [[lm.x, lm.y, lm.z] for lm in estimation.pose_world_landmarks[0]]
    if estimation and estimation.pose_world_landmarks
    else [[0, 0, 0]] * N_LANDMARKS
    for estimation in estimations
  ]

  t0 = videosource.buffer_timestamps[0]
  for frame_idx, landmarks in enumerate(world_estimations):
    timestamp = (videosource.buffer_timestamps[frame_idx] - t0) / 1e9
    flat = [v for lm in landmarks for v in lm]
    writer.writerow([path.name, frame_idx, round(timestamp, 4), *flat])

  print(path.name, len(world_estimations), "frames")

estimation_output_file.close()

deadlift_1.mp4 150 frames
deadlift_10.mp4 150 frames
deadlift_11.mp4 121 frames
deadlift_12.mp4 94 frames
deadlift_13.mp4 263 frames
deadlift_14.mp4 150 frames
deadlift_15.mp4 68 frames
deadlift_16.mp4 109 frames
deadlift_17.mp4 147 frames
deadlift_18.mp4 105 frames
deadlift_19.mp4 62 frames
deadlift_2.mp4 150 frames
deadlift_20.mp4 59 frames
deadlift_21.mp4 195 frames
deadlift_22.mp4 70 frames
deadlift_23.mp4 72 frames
deadlift_24.mp4 103 frames
deadlift_25.mp4 938 frames
deadlift_26.mp4 180 frames
deadlift_27.mp4 165 frames
deadlift_28.mp4 192 frames
deadlift_29.mp4 103 frames
deadlift_3.mp4 150 frames
deadlift_30.mp4 185 frames
deadlift_31.mp4 144 frames
deadlift_32.mp4 105 frames
deadlift_4.mp4 150 frames
deadlift_5.mp4 150 frames
deadlift_6.mp4 150 frames
deadlift_7.mp4 150 frames
deadlift_8.mp4 150 frames
deadlift_9.mp4 147 frames


In [31]:
columns = ["video", "frame", "timestamp"] + [
    f"{axis}{i}" for i in range(N_LANDMARKS) for axis in "xyz"
]

df = pd.read_csv("deadlift-estimations.csv", header=None, names=columns)

In [32]:
df.head()

,video,frame,timestamp,x0,y0,z0,x1,y1,z1,x2,...,z29,x30,y30,z30,x31,y31,z31,x32,y32,z32
0,deadlift_1.mp4,0,0.0000,-0.032239,-0.183095,-0.725946,-0.017194,-0.223422,-0.719837,-0.016311,...,-0.013437,-0.133350,0.563342,-0.034296,0.112871,0.623815,-0.120198,-0.160220,0.569796,-0.160483
1,deadlift_1.mp4,1,0.0093,-0.032240,-0.183366,-0.725997,-0.017188,-0.223636,-0.719869,-0.016304,...,-0.012488,-0.133075,0.563373,-0.033439,0.112570,0.626080,-0.119316,-0.160146,0.570099,-0.158913
2,deadlift_1.mp4,2,0.0177,-0.032228,-0.183271,-0.726201,-0.017154,-0.223501,-0.720079,-0.016270,...,-0.009922,-0.133037,0.563394,-0.032214,0.112541,0.627485,-0.115955,-0.160252,0.570349,-0.156808
3,deadlift_1.mp4,3,0.0210,-0.031824,-0.183371,-0.726222,-0.016855,-0.223602,-0.720084,-0.015966,...,-0.009709,-0.132898,0.563455,-0.032172,0.112504,0.628250,-0.115840,-0.160251,0.570456,-0.156732
4,deadlift_1.mp4,4,0.0295,-0.031019,-0.183351,-0.726355,-0.016135,-0.223569,-0.720163,-0.015240,...,-0.007933,-0.132792,0.563479,-0.032120,0.112958,0.629106,-0.113618,-0.160268,0.570924,-0.156376


In [35]:
df.describe()

,frame,timestamp,x0,y0,z0,x1,y1,z1,x2,y2,...,z29,x30,y30,z30,x31,y31,z31,x32,y32,z32
count,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,...,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000
mean,146.502633,1.351336,0.064891,-0.380193,-0.353680,0.074764,-0.407944,-0.359691,0.075822,-0.405568,...,0.181246,-0.080395,0.636991,0.091767,0.088586,0.665373,0.148703,-0.075604,0.648767,0.039641
std,196.737222,1.726701,0.291857,0.203550,0.231537,0.287922,0.199420,0.226549,0.286874,0.200109,...,0.174184,0.143151,0.150440,0.149465,0.186313,0.156566,0.205139,0.210379,0.145246,0.176565
min,0.000000,0.000000,-0.637806,-0.678592,-0.780149,-0.639778,-0.686138,-0.788315,-0.639323,-0.685828,...,-0.177714,-0.866092,-0.246839,-0.650136,-0.549251,-0.207349,-0.270884,-0.866937,-0.295581,-0.693617
25%,40.000000,0.310150,-0.038831,-0.573373,-0.514328,-0.020953,-0.596013,-0.513141,-0.019243,-0.594887,...,0.029544,-0.146826,0.556605,-0.026951,-0.031682,0.601203,-0.030847,-0.203720,0.583309,-0.105423
50%,82.000000,0.651700,0.018939,-0.392320,-0.395296,0.028965,-0.424033,-0.396716,0.030518,-0.420358,...,0.186390,-0.098801,0.694540,0.082591,0.114807,0.714572,0.152237,-0.108602,0.697314,0.050971
75%,138.000000,1.385750,0.268706,-0.199825,-0.176666,0.267476,-0.234025,-0.181382,0.268042,-0.231642,...,0.312690,0.000000,0.739158,0.200345,0.189902,0.778588,0.335661,0.090969,0.748233,0.152438
max,937.000000,7.320800,0.577612,0.536104,0.332215,0.587770,0.554523,0.328025,0.587842,0.556147,...,0.736192,0.482756,0.833126,0.712461,0.470542,0.878429,0.750258,0.570171,0.832639,0.692530
